Morganの処理：まずVarianceThresholdで分散ゼロ（QM9では大半が該当するはず）のビットを除去。残ったビットのうち重要度上位を維持しつつ、全体をSVD（もしくはUMAP）で数十次元に圧縮し、圧縮後の特徴量として追加する

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

SMILES_COL = "smiles"
LABEL_COL = "gap_eV"

df = pd.read_csv("../train/qm9_morgan_fp.csv")
X = df.drop(columns=[SMILES_COL, LABEL_COL])
y = df[LABEL_COL]
print(df.shape)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
importances = np.zeros(X.shape[1])

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    pred = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, pred)
    fold_scores.append(mae)
    importances += model.feature_importances_ / 5
    print(f"Fold {fold}: MAE = {mae:.4f}")

print(f"\nMorgan CV MAE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances,
}).sort_values("importance", ascending=False)

importance_df.head(20)
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

importance_df.to_csv(RESULTS_DIR / "morgan_importance.csv", index=False)
print(f"保存しました: {RESULTS_DIR / 'morgan_importance.csv'}")

(106952, 2050)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.170072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4096
[LightGBM] [Info] Number of data points in the train set: 85561, number of used features: 2048
[LightGBM] [Info] Start training from score 6.831522
Fold 1: MAE = 0.2915
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.190901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4096
[LightGBM] [Info] Number of data points in the train set: 85561, number of used features: 2048
[LightGBM] [Info] Start training from score 6.839479
Fold 2: MAE = 0.2960
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.175062 seconds.
You can set `force_row_

重要でない特徴量の除去

In [ ]:
import pandas as pd

# 重要度0のビットを特定
importance_df = pd.read_csv("results/morgan_importance.csv")
zero_importance_features = importance_df[importance_df["importance"] == 0]["feature"].tolist()

print(f"重要度0のビット数: {len(zero_importance_features)}")

# Morganの生データから、重要度0のビットを除去
df = pd.read_csv("../train/qm9_morgan_fp.csv")
kept_columns = [c for c in df.columns if c not in zero_importance_features]

df_filtered = df[kept_columns]
df_filtered.to_csv("result/qm9_morgan_fp_filtered.csv", index=False)

print(f"元の列数: {df.shape[1] - 2}")  # smiles, gap_eVを除いた特徴量数
print(f"除去した列数: {len(zero_importance_features)}")
print(f"残った列数: {df_filtered.shape[1] - 2}")

重要度0のビット数: 686


OSError: Cannot save file into a non-existent directory: 'result'